# Клиенты, повторные контакты и перезвон — Левин
Продолжение [model_levin.ipynb](model_levin.ipynb). Поводом стал вопрос: *одна ли строка на клиента?* Оказалось, **нет** — один клиент может встречаться в нескольких кампаниях. Отсюда три задачи:

1. **Восстановить клиентов** (в данных нет id) и проверить качество связок.
2. **Пересчитать валидацию с учётом клиентов:** не должно быть клиента одновременно в train и test.
3. **Модель перезвона:** для клиентов с историей можно взять признаки их *прошлой* записи (длительность прошлого звонка, итог, баланс) — это законные признаки без утечки.

Попутно проверяем подозрительный признак `campaign` (число звонков в текущей кампании).

## 0. Импорты и данные

In [1]:
import warnings; warnings.filterwarnings('ignore')
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
from collections import defaultdict
from scipy.sparse import coo_matrix
from scipy.sparse.csgraph import connected_components
import utils_levin as ul

sns.set_theme(style='whitegrid'); pd.set_option('display.width', 140, 'display.max_columns', 40)
raw, y, year, period = ul.load()
d = raw.copy()                                   # рабочая копия с датой
d['date'] = pd.to_datetime(dict(year=year, month=raw.month.map(ul.MONTHS), day=raw.day_of_week))
for c in ['job', 'education']: d[c] = d[c].fillna('unknown')
d['y01'] = y
print(d.shape, '| период:', d.date.min().date(), '-', d.date.max().date())

(45211, 19) | период: 2008-05-05 - 2010-11-17


## 1. Восстановление клиентов
**Идея.** Если у строки `pdays = N`, то прошлый контакт был ровно за N дней до неё. Ищем более раннюю запись на эту дату с тем же профилем: `job`, `marital`, `education`, `default` и возраст (мог вырасти не более чем на число прошедших лет + 1).

**Проверка связки (без таргета):** у ранней записи `campaign + previous` (все её контакты) должно равняться `previous` поздней записи.

Уровни надёжности:
- **A** — единственный кандидат, проверка `previous` пройдена;
- **B** — кандидатов несколько, проверку проходит ровно один;
- **D** — несколько прошли проверку, берём ближайшего по балансу;
- **W** — единственный кандидат, но проверка не пройдена (слабая связка);
- **X** — кандидатов несколько, ни один не прошёл проверку (не связываем);
- **нет** — кандидатов нет (прошлый контакт вне датасета или профиль изменился).

Для разбиения на группы (клиентов) неоднозначные случаи D и X склеиваем со *всеми* кандидатами — это консервативно: лучше объединить лишнего, чем оставить одного клиента в train и test.

In [2]:
def link_clients(d):
    keys = defaultdict(list)
    for i, k in enumerate(zip(d.date, d.job, d.marital, d.education, d.default)): keys[k].append(i)
    n = len(d); prev = np.full(n, -1); tier = np.full(n, '', dtype=object); edges = []
    for i in np.where(d.pdays.values > 0)[0]:
        gap = int(d.pdays.iat[i]); dt = d.date.iat[i] - pd.Timedelta(days=gap)
        cand = [j for j in keys.get((dt, d.job.iat[i], d.marital.iat[i], d.education.iat[i], d.default.iat[i]), [])
                if j < i and 0 <= d.age.iat[i] - d.age.iat[j] <= gap // 365 + 1]
        if not cand: tier[i] = 'нет'; continue
        ok = [j for j in cand if d.campaign.iat[j] + d.previous.iat[j] == d.previous.iat[i]]   # проверка по числу контактов
        if len(cand) == 1: prev[i], tier[i] = cand[0], ('A' if ok else 'W')
        elif len(ok) == 1: prev[i], tier[i] = ok[0], 'B'
        elif len(ok) > 1: prev[i], tier[i] = min(ok, key=lambda j: abs(d.balance.iat[j] - d.balance.iat[i])), 'D'
        else: tier[i] = 'X'
        edges += [(i, j) for j in (cand if tier[i] in ('D', 'X') else [prev[i]])]
    return prev, tier, np.array(edges)

prev, tier, edges = link_clients(d)
n = len(d)
_, client = connected_components(coo_matrix((np.ones(len(edges)), (edges[:, 0], edges[:, 1])), shape=(n, n)), directed=False)
depth = np.zeros(n, int)                               # сколько прошлых кампаний в цепочке до этой записи
for i in np.where(prev >= 0)[0]: depth[i] = depth[prev[i]] + 1

t = pd.Series(tier[d.pdays.values > 0]).value_counts().reindex(['A', 'B', 'D', 'W', 'X', 'нет'])
print('Строк с историей (pdays>0):', (d.pdays > 0).sum()); display(t.to_frame('строк').assign(доля=lambda x: (x['строк'] / x['строк'].sum()).round(3)).T)

Строк с историей (pdays>0): 8257


,A,B,D,W,X,нет
строк,3718.00,1572.00,1467.000,266.000,254.000,980.000
доля,0.45,0.19,0.178,0.032,0.031,0.119


In [3]:
# Качество связок: итог прошлой записи (y_old) должен совпадать с poutcome новой записи
linked = np.where(np.isin(tier, ['A', 'B', 'D', 'W']))[0]
chk = pd.crosstab(d.poutcome.iloc[linked].fillna('NaN').values, d.y01.values[prev[linked]], rownames=['poutcome (новая)'], colnames=['y (старая)'])
display(chk)
for tr in ['A', 'B', 'D', 'W']:
    ii = np.where(tier == tr)[0]; yo = d.y01.values[prev[ii]]; po = d.poutcome.iloc[ii].values
    agree = np.mean([(p == 'success') == (o == 1) for p, o in zip(po, yo) if p in ('success', 'failure')])
    print(f'уровень {tr}: связок {len(ii):5d}; success<->yes / failure<->no согласуются в {agree:.1%}; корреляция баланса {np.corrcoef(d.balance.values[ii], d.balance.values[prev[ii]])[0, 1]:.2f}')

y (старая),0,1
poutcome (новая),,
failure,4901,0
other,573,38
success,1,1510


уровень A: связок  3718; success<->yes / failure<->no согласуются в 100.0%; корреляция баланса 0.98
уровень B: связок  1572; success<->yes / failure<->no согласуются в 100.0%; корреляция баланса 0.90
уровень D: связок  1467; success<->yes / failure<->no согласуются в 99.9%; корреляция баланса 0.98
уровень W: связок   266; success<->yes / failure<->no согласуются в nan%; корреляция баланса 0.03


In [4]:
sizes = pd.Series(client).value_counts()
print(f'Клиентов (групп): {len(sizes)} на {n} строк; строк в группах из 2+ записей: {(sizes[sizes > 1]).sum()} ({(sizes[sizes > 1]).sum() / n:.1%})')
print('Размеры групп:', sizes.value_counts().sort_index().head(6).to_dict(), '| максимум:', sizes.max())

Клиентов (групп): 33690 на 45211 строк; строк в группах из 2+ записей: 16409 (36.3%)
Размеры групп: {1: 28802, 2: 3279, 3: 699, 4: 310, 5: 177, 6: 118} | максимум: 416


**Вывод.** Связки надёжные: у уровней A/B/D почти идеальное согласие `poutcome` ↔ прошлый `y`, баланс сильно коррелирует. Часть строк с историей связать не удалось (профиль изменился или прошлый контакт вне датасета) — это нормально, мы теряем часть истории, но не создаём ложных связей. Заметная доля строк принадлежит клиентам с 2+ записями → валидация должна это учитывать.

## 2. Валидация с учётом клиентов
Финальная модель из `model_levin.ipynb` (LightGBM, фичи без `duration` и прокси периода). Сравниваем схемы:
- **strat** — как раньше (случайные фолды) и **strat grouped** (`StratifiedGroupKFold`: клиент целиком в одном фолде);
- **block** — как раньше и **block purged** (из train убраны клиенты, которые есть в test);
- **time** — без изменений: прошлая запись в train, будущая в test — это ровно жизненная ситуация.

In [5]:
X = ul.build(raw, drop=ul.NP_DROP)
sch = {'strat (как раньше)': ul.strat_splits(y), 'strat grouped': ul.strat_splits(y, client),
       'block (как раньше)': ul.block_splits(period, y), 'block purged': ul.block_splits(period, y, client),
       'time': ul.time_splits(n)}
rows = []
for k, sp in sch.items():
    m = ul.run_cv(ul.make_lgb, X, y, sp)[0].mean()
    rows.append({'схема': k, 'AUC': m.auc, 'PR-AUC': m.pr, 'Lift@10%': m.lift10, 'train, строк': np.mean([len(a) for a, _ in sp])})
res2 = pd.DataFrame(rows).set_index('схема'); res2.round(3)

,AUC,PR-AUC,Lift@10%,"train, строк"
схема,,,,
strat (как раньше),0.744,0.383,3.824,36168.8
strat grouped,0.743,0.383,3.803,36168.8
block (как раньше),0.723,0.371,3.775,36168.8
block purged,0.728,0.372,3.827,30914.6
time,0.602,0.194,1.514,22606.0


In [6]:
# Чувствительность: «точные» группы — только выбранные связки A, B, D (без склейки неоднозначных X и слабых W)
e2 = np.array([(i, prev[i]) for i in np.where(np.isin(tier, ['A', 'B', 'D']))[0]])
_, client2 = connected_components(coo_matrix((np.ones(len(e2)), (e2[:, 0], e2[:, 1])), shape=(n, n)), directed=False)
s2 = pd.Series(client2).value_counts()
print(f'Точные группы: {len(s2)}; строк в группах из 2+ записей: {s2[s2 > 1].sum() / n:.1%}; максимальная группа: {s2.max()}')
rows = []
for k, sp in {'strat grouped (точные группы)': ul.strat_splits(y, client2), 'block purged (точные группы)': ul.block_splits(period, y, client2)}.items():
    m = ul.run_cv(ul.make_lgb, X, y, sp)[0].mean(); rows.append({'схема': k, 'AUC': m.auc, 'PR-AUC': m.pr, 'Lift@10%': m.lift10, 'train, строк': np.mean([len(a) for a, _ in sp])})
pd.concat([res2, pd.DataFrame(rows).set_index('схема')]).round(3)

Точные группы: 38454; строк в группах из 2+ записей: 27.3%; максимальная группа: 8


,AUC,PR-AUC,Lift@10%,"train, строк"
схема,,,,
strat (как раньше),0.744,0.383,3.824,36168.8
strat grouped,0.743,0.383,3.803,36168.8
block (как раньше),0.723,0.371,3.775,36168.8
block purged,0.728,0.372,3.827,30914.6
time,0.602,0.194,1.514,22606.0
strat grouped (точные группы),0.745,0.384,3.798,36168.8
block purged (точные группы),0.727,0.372,3.799,33491.8


**Как читать:** если «grouped/purged» заметно хуже «как раньше», значит, модель выигрывала от того, что один клиент был и в train, и в test. Если разница в пределах шума (~0.005) — наши прежние оценки честные.

## 3. Проверка `campaign` — не утечка ли из будущего?
`campaign` — число контактов в текущей кампании, **включая последний звонок**, закончившийся результатом. Клиент, который согласился, больше не звонят; тем, кто отказывается, звонят снова и снова. То есть частично это следствие итога. До первого звонка кампании этот признак неизвестен. Проверим, сколько он даёт.

In [7]:
Xc0 = X.drop(columns=['campaign', 'campaign_clip'])
rows = []
for k in ['strat grouped', 'block purged', 'time']:
    a, b = (ul.run_cv(ul.make_lgb, Z, y, sch[k])[0].mean() for Z in (X, Xc0))
    rows.append({'схема': k, 'AUC с campaign': a.auc, 'AUC без campaign': b.auc, 'Δ': b.auc - a.auc, 'Lift@10% с': a.lift10, 'Lift@10% без': b.lift10})
pd.DataFrame(rows).set_index('схема').round(3)

,AUC с campaign,AUC без campaign,Δ,Lift@10% с,Lift@10% без
схема,,,,,
strat grouped,0.743,0.738,-0.006,3.803,3.794
block purged,0.728,0.724,-0.005,3.827,3.805
time,0.602,0.604,0.002,1.514,1.626


## 4. Модель перезвона: клиенты с историей
Берём строки с **надёжной** связкой (A и B). Для каждой подтягиваем признаки *прошлой* записи того же клиента:

| Признак | Смысл |
|---|---|
| `prev_duration` | длительность прошлого звонка (интерес клиента) |
| `prev_campaign` | сколько раз звонили в прошлой кампании |
| `prev_dur_per_call` | длительность на один звонок |
| `prev_balance_log`, `balance_delta` | прошлый баланс и его изменение |
| `prev_depth` | сколько кампаний было до прошлой |
| `prev_contact` | канал прошлого контакта |

Итог прошлой кампании уже есть в `poutcome`. Все эти данные известны к моменту решения о перезвоне — утечки нет.
Валидация: **strat grouped**, **block purged**, **time** (на когорте).

In [8]:
ci = np.where(np.isin(tier, ['A', 'B']))[0]             # когорта: строки с надёжной связкой (по возрасту строки = хронологически)
P = d.iloc[prev[ci]]; sl = lambda b: np.sign(b) * np.log1p(np.abs(b))
F = pd.DataFrame({'prev_duration': P.duration.values, 'prev_campaign': P.campaign.values,
                  'prev_dur_per_call': P.duration.values / P.campaign.values,
                  'prev_balance_log': sl(P.balance.values), 'balance_delta': sl(d.balance.values[ci]) - sl(P.balance.values),
                  'prev_depth': depth[prev[ci]], 'prev_contact': P.contact.fillna('unknown').astype('category').values})
Xb = X.iloc[ci].reset_index(drop=True); yc, gc, pc = y[ci], client[ci], period[ci]
print(f'Когорта: {len(ci)} строк; доля yes {yc.mean():.3f} (в целом по данным {y.mean():.3f}); клиентов {len(set(gc))}')
sch_c = {'strat grouped': ul.strat_splits(yc, gc), 'block purged': ul.block_splits(pc, yc, gc), 'time': ul.time_splits(len(yc))}
mk_c = lambda Z: ul.make_lgb(Z, num_leaves=4, n_estimators=250, learning_rate=0.03, min_child_samples=30, subsample=0.8, colsample_bytree=0.7, reg_lambda=5)

Когорта: 5290 строк; доля yes 0.274 (в целом по данным 0.117); клиентов 4116


In [9]:
sets = {'база (без прошлой записи)': [], '+ prev_duration': ['prev_duration'], '+ prev_campaign': ['prev_campaign'],
        '+ баланс прошлый и сдвиг': ['prev_balance_log', 'balance_delta'], '+ prev_depth': ['prev_depth'], '+ prev_contact': ['prev_contact'],
        '+ все признаки прошлой записи': list(F.columns)}
rows = []
for name, add in sets.items():
    Z = pd.concat([Xb, F[add]], axis=1)
    r = {'набор': name}
    for k, sp in sch_c.items(): m = ul.run_cv(mk_c, Z, yc, sp)[0].mean(); r[f'{k} AUC'] = m.auc; r[f'{k} lift10'] = m.lift10
    rows.append(r)
abl = pd.DataFrame(rows).set_index('набор')
auc_cols = [c for c in abl if 'AUC' in c]
pd.concat([abl[auc_cols].round(3), abl[auc_cols].sub(abl[auc_cols].iloc[0]).add_prefix('Δ ').round(3), abl[[c for c in abl if 'lift' in c]].round(2)], axis=1)

,strat grouped AUC,block purged AUC,time AUC,Δ strat grouped AUC,Δ block purged AUC,Δ time AUC,strat grouped lift10,block purged lift10,time lift10
набор,,,,,,,,,
база (без прошлой записи),0.840,0.772,0.704,0.000,0.000,0.000,2.81,2.75,2.16
+ prev_duration,0.840,0.770,0.705,-0.000,-0.002,0.001,2.84,2.71,2.14
+ prev_campaign,0.841,0.772,0.702,0.001,0.000,-0.002,2.84,2.77,2.13
+ баланс прошлый и сдвиг,0.840,0.772,0.702,-0.000,-0.000,-0.002,2.83,2.75,2.17
+ prev_depth,0.843,0.772,0.703,0.003,0.001,-0.001,2.91,2.72,2.15
+ prev_contact,0.848,0.778,0.706,0.008,0.006,0.002,2.80,2.68,2.15
+ все признаки прошлой записи,0.850,0.778,0.705,0.009,0.006,0.002,2.92,2.74,2.12


**Как читать:** `Δ` — прирост AUC относительно базы на той же когорте. Показываем, дают ли признаки прошлого звонка что-то сверх `poutcome`.

In [10]:
# Конверсия по длительности прошлого звонка и итогу прошлой кампании (простой взгляд на данные)
tmp = pd.DataFrame({'prev_y': P.y01.values, 'prev_duration': P.duration.values, 'y': yc})
tmp['прошлый звонок'] = pd.qcut(tmp.prev_duration, 4, labels=['короткий', 'средний-', 'средний+', 'длинный'])
tmp.pivot_table(index='прошлый звонок', columns='prev_y', values='y', aggfunc=['mean', 'size'], observed=True).round(3)

mean         size     
prev_y              0      1     0    1
прошлый звонок                         
короткий        0.139  0.674  1241   86
средний-        0.137  0.728  1035  287
средний+        0.113  0.723   901  423
длинный         0.150  0.588   715  602

## 5. Повторная покупка: клиенты, согласившиеся в прошлый раз
Самое близкое к вопросу «сделают ли они ещё покупку»: клиенты, у которых прошлый итог — **yes**. Вопрос: можно ли среди них отличить тех, кто купит снова.

In [11]:
sub = np.where(P.y01.values == 1)[0]
Xs, Fs, ys, gs = Xb.iloc[sub].reset_index(drop=True), F.iloc[sub].reset_index(drop=True), yc[sub], gc[sub]
print(f'Купили в прошлый раз: {len(sub)} клиентов; купили снова: {ys.mean():.1%}')
sp = ul.strat_splits(ys, gs)
mk_s = lambda Z: ul.make_lgb(Z, num_leaves=3, n_estimators=150, learning_rate=0.03, min_child_samples=20, subsample=0.8, colsample_bytree=0.7, reg_lambda=10)
for name, Z in [('без прошлой записи', Xs), ('+ признаки прошлой записи', pd.concat([Xs, Fs], axis=1))]:
    m = ul.run_cv(mk_s, Z, ys, sp)[0].mean(); print(f'{name:28s}: AUC {m.auc:.3f}  Lift@10% {m.lift10:.2f}')

Купили в прошлый раз: 1398 клиентов; купили снова: 66.3%


без прошлой записи          : AUC 0.655  Lift@10% 1.20


+ признаки прошлой записи   : AUC 0.663  Lift@10% 1.29


## 6. Порог и цена ошибки для перезвона
Когорта конвертируется заметно лучше, поэтому «звонить всем» выгодно уже при `r/c` ≈ 1/доля_yes. Протокол как раньше: порог выбираем по OOF на train (клиенты не пересекаются), оцениваем на test (стратифицированный по клиентам hold-out 20%).

In [12]:
best_name = max(sets, key=lambda k: (abl.loc[k, 'block purged AUC'] + abl.loc[k, 'time AUC']) / 2)
Zc = pd.concat([Xb, F[sets[best_name]]], axis=1); print('Набор для финальной модели перезвона:', best_name)
tr0, te0 = ul.strat_splits(yc, gc)[0]
Ztr, Zte, ytr, yte, gtr = Zc.iloc[tr0], Zc.iloc[te0], yc[tr0], yc[te0], gc[tr0]
oof = np.zeros(len(ytr))
for a, b in ul.strat_splits(ytr, gtr): oof[b] = mk_c(Ztr).fit(Ztr.iloc[a], ytr[a]).predict_proba(Ztr.iloc[b])[:, 1]
p_te = mk_c(Ztr).fit(Ztr, ytr).predict_proba(Zte)[:, 1]
from sklearn.metrics import roc_auc_score, average_precision_score, confusion_matrix
print(f'Hold-out: n={len(yte)}, доля yes {yte.mean():.3f}; AUC {roc_auc_score(yte, p_te):.3f}, PR-AUC {average_precision_score(yte, p_te):.3f}, Lift@10% {ul.lift_at(yte, p_te):.2f}')

ts = np.linspace(0.05, 0.9, 86)
profit = lambda p, yy, t, r, c=1.0: ((p >= t) * (r * yy - c)).sum() / len(yy) * 1000      # прибыль на 1000 клиентов
rows = []
for r in (3, 5, 8):
    t_star = ts[np.argmax([profit(oof, ytr, t, r) for t in ts])]; sel = p_te >= t_star; tn, fp, fn, tp = confusion_matrix(yte, sel).ravel()
    rows.append({'r/c': r, 'порог': round(t_star, 2), 'звоним, %': 100 * sel.mean(), 'precision': tp / max(tp + fp, 1), 'recall': tp / (tp + fn),
                 'прибыль модели': profit(p_te, yte, t_star, r), 'звонить всем': profit(np.ones(len(yte)), yte, .5, r)})
pd.DataFrame(rows).set_index('r/c').round(3)

Набор для финальной модели перезвона: + все признаки прошлой записи


Hold-out: n=1058, доля yes 0.274; AUC 0.878, PR-AUC 0.725, Lift@10% 3.02


,порог,"звоним, %",precision,recall,прибыль модели,звонить всем
r/c,,,,,,
3,0.40,27.788,0.690,0.700,297.732,-177.694
5,0.19,45.463,0.541,0.897,774.102,370.510
8,0.13,56.427,0.454,0.934,1484.877,1192.817


In [13]:
# Lift по децилям (OOF grouped-CV на всей когорте)
oof_all = ul.run_cv(mk_c, Zc, yc, ul.strat_splits(yc, gc))[1]
dd = pd.DataFrame({'p': oof_all, 'y': yc}); dd['дециль'] = pd.qcut(dd.p.rank(method='first', ascending=False), 10, labels=range(1, 11))
g = dd.groupby('дециль', observed=True).y.agg(['mean', 'sum']); g['lift'] = g['mean'] / yc.mean(); g['накоп. доля yes'] = g['sum'].cumsum() / yc.sum(); g.round(3)

,mean,sum,lift,накоп. доля yes
дециль,,,,
1,0.796,421,2.903,0.290
2,0.660,349,2.407,0.531
3,0.427,226,1.559,0.687
4,0.316,167,1.152,0.802
5,0.210,111,0.766,0.879
6,0.115,61,0.421,0.921
7,0.070,37,0.255,0.946
8,0.055,29,0.200,0.966
9,0.047,25,0.172,0.983


## 7. Вердикт

**1. Клиенты повторяются, и мы их восстановили.** Из 8 257 строк с историей надёжно связаны 5 290 (уровни A+B), ещё 1 467 — уровень D (ближайший по балансу). Качество связок: итог прошлой записи совпадает с `poutcome` новой в 99.9–100% случаев, баланс коррелирует на 0.90–0.98. Не связаны 980 (кандидатов нет), 254 (неоднозначно) и 266 слабых (W, баланс не коррелирует — не используем). По «точным» группам ≈ **27% строк** принадлежат клиентам с 2+ записями (до 36% при консервативной склейке).

**2. Валидация с учётом клиентов не изменила оценок** — прежние результаты честные:

| Схема | как раньше | с учётом клиентов | точные группы |
|---|---|---|---|
| strat, AUC | 0.744 | 0.743 | 0.745 |
| block, AUC | 0.723 | 0.728 | 0.727 |
| time, AUC | 0.602 | — (не нужно) | — |

Финальная модель (мелкие деревья, сильная регуляризация) не запоминает отдельных клиентов. Схема `time` от дублей защищена по построению.

**3. `campaign` — формально «из будущего»** (включает последний звонок, закончившийся итогом), но вклад мизерный: без него AUC −0.005/−0.006 (strat grouped / block purged) и +0.002 на time, Lift@10% почти не меняется. **Рекомендация: убрать из боевой модели** — цена пренебрежимо мала, зато признак корректен на момент решения. (В `model_levin.ipynb` он пока есть.)

**4. Перезвон клиентам с историей (когорта 5 290 строк, конверсия 27.4% против 11.7% в целом).**

| Схема | AUC на когорте (база: фичи основной модели) | + признаки прошлой записи |
|---|---|---|
| strat grouped | 0.840 | 0.850 |
| block purged | 0.772 | 0.778 |
| time | 0.704 | 0.705 |

Когорта предсказывается заметно лучше, чем весь датасет (0.77 против 0.73 на новых месяцах), но **почти вся сила — в `poutcome`** (итог прошлой кампании). Признаки прошлой записи дают +0.002…0.009 AUC, то есть практически ничего:
- `prev_duration`, `prev_campaign`, баланс — прирост ≈ 0;
- небольшой прирост у `prev_contact`, но, вероятно, это прокси периода прошлого контакта (`unknown` = май–июнь 2008), а не поведение клиента.

**Гипотеза «долгий прошлый разговор → клиент снова купит» не подтвердилась.** Конверсия по длительности прошлого звонка: у прошлых отказавшихся ≈ 11–15% во всех квартилях, у прошлых купивших 59–73% и немонотонно (самая длинная группа даже ниже: 59%).

**5. Повторная покупка** (1 398 клиентов, купивших в прошлый раз; снова купили **66.3%**): AUC 0.655 → 0.663 с признаками прошлой записи, Lift@10% 1.2–1.3 (максимально возможный при такой доле ≈ 1.5). Отличить будущих повторных покупателей среди прошлых покупателей почти нечем: данных о клиенте мало.

**6. Деньги и порог (иллюстративные `r/c`; на hold-out 20% с разбиением по клиентам).** Hold-out: AUC 0.878, PR-AUC 0.725, Lift@10% 3.0 — это случайный split, **оптимистично** (честный диапазон по block/time: 0.70–0.78). Топ-10% когорты по модели конвертируются в ≈ 80%, два верхних дециля покрывают ≈ 53% всех «да».

| `r/c` | порог | звоним, % | precision | recall | прибыль / 1000 | звонить всем |
|---|---|---|---|---|---|---|
| 3 | 0.40 | 28 | 0.69 | 0.70 | 298 | −178 |
| 5 | 0.19 | 45 | 0.54 | 0.90 | 774 | 371 |
| 8 | 0.13 | 56 | 0.45 | 0.93 | 1485 | 1193 |

В когорте «звонить всем» выгодно уже при `r/c` > ≈ 3.6 (1/0.274), поэтому выигрыш модели над «звонить всем» заметен лишь при невысоком `r/c`.

### Практические выводы
- **Холодный обзвон** (клиенты без истории): модель слабая, AUC ≈ 0.65 на новых месяцах.
- **Перезвон клиентам с историей**: сильный сегмент; главный сигнал — итог прошлой кампании. Отдельная модель даёт AUC ≈ 0.77 на новых периодах и позволяет отсечь явных «не купит» (децили 6–10 дают 4–12% конверсии).
- **Сложные признаки прошлой записи не нужны:** на этих данных они ≈ не улучшают результат.

### Ограничения
- Связки приблизительные (по профилю); 15% строк с историей связать не удалось, а часть клиентов, вероятно, сменила профиль (например, семейное положение).
- Повторная покупка — это тот же депозит в новой кампании; данных о других продуктах нет.
- `r` и `c` иллюстративные; hold-out-оценки перезвона оптимистичны (случайный split).
